In [1]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

# 0. Preparation
###
# Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [2]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Reading data file from GoogleDrive
df = pd.read_pickle("/content/drive/My Drive/Team Project X-Rays/Dataframes/df_basic_1.4.pkl")
df.head()

# Define data name
df_name = "Baseline 1.4"

# Define random subsample for computation efficiency
#df = df.sample(500)


In [14]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Show labels
print(df.Case.value_counts())

# Check distributions after normalisation
df.iloc[:,550:557].describe()



<class 'pandas.core.frame.DataFrame'>
(21105, 4098)
Case
0    10191
2     6012
1     3564
3     1338
Name: count, dtype: int64


,PX_549,PX_550,PX_551,PX_552,PX_553,PX_554,PX_555
count,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.00000,21105.000000
mean,133.749017,127.038048,120.961952,116.222886,113.078654,111.27022,110.286709
std,39.514259,40.560074,41.328429,41.812573,42.479271,43.12433,43.900381
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
25%,108.000000,100.000000,93.000000,87.000000,84.000000,81.00000,80.000000
50%,135.000000,128.000000,122.000000,117.000000,113.000000,111.00000,110.000000
75%,162.000000,157.000000,151.000000,146.000000,143.000000,142.00000,141.000000
max,250.000000,249.000000,249.000000,248.000000,255.000000,247.00000,248.000000


In [9]:
# Check missing values
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))

# No missing vars. We can continue the ML modelling.

<class 'pandas.core.frame.DataFrame'>
Index: 21105 entries, 0 to 21164
Columns: 4098 entries, Name to PX_4096
dtypes: object(2), uint8(4096)
memory usage: 83.4+ MB
None
Missing vars in columns:
 Name       0
Case       0
PX_1       0
PX_2       0
PX_3       0
          ..
PX_4092    0
PX_4093    0
PX_4094    0
PX_4095    0
PX_4096    0
Length: 4098, dtype: int64
Number of total missing vars: 0
Number of total missing vars (% of all obs): 0.0


In [10]:
# 1. Data preprocessing
###

# Create categorical variable from Case
df["Case"] = df.Case.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
df.Case.astype(int)

# Check construction
print(df.Case.value_counts())

# Split data into target and features
target = df.Case

# Features data: Drop Names and target
data = df.drop(["Name", "Case"], axis = 1)
data.head()
data.shape

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

Case
0    10191
2     6012
1     3564
3     1338
Name: count, dtype: int64


In [11]:
# Standardisation of mean and variance with StandardScaler()
###

# Instantiate StandardScaler function
sc = StandardScaler()

# Standardize the data on the training set first and store the mean and std (scaler.fit_transform)
X_train = sc.fit_transform(X_train)

# Then apply the mean and std from the training data to standardize the test data (scaler.transform)
# This prevents the model to learn from the test data but only from the train data (avoid data leakage)
X_test = sc.transform(X_test)


In [18]:
# Convert to DataFrame
X_train_df = pd.DataFrame(X_train)

# Select columns 550 to 557
columns_to_analyze = X_train_df.iloc[:, 550:558]

# Calculate statistics
mean = columns_to_analyze.mean()
std = columns_to_analyze.std()
median = columns_to_analyze.median()
count = columns_to_analyze.count()

# Combine results into a DataFrame and transpose it for row display
statistics_df = pd.DataFrame({
    'Mean': mean,
    'Standard Deviation': std,
    'Median': median,
    'Count': count
}).T  # Transpose to make statistics as rows

# Set column names for better readability
statistics_df.columns = columns_to_analyze.columns

# Display the statistics DataFrame
print(statistics_df)

                             550           551           552           553  \
Mean                1.369827e-16  5.260474e-19  7.049035e-17  1.893771e-18   
Standard Deviation  1.000030e+00  1.000030e+00  1.000030e+00  1.000030e+00   
Median              1.887491e-02  1.113380e-02  1.427432e-02 -1.432022e-02   
Count               1.688400e+04  1.688400e+04  1.688400e+04  1.688400e+04   

                             554           555           556           557  
Mean               -1.281451e-16  5.344642e-17 -2.283046e-17  1.485558e-16  
Standard Deviation  1.000030e+00  1.000030e+00  1.000030e+00  1.000030e+00  
Median             -1.218761e-02  2.509244e-03  1.569604e-03  2.117075e-02  
Count               1.688400e+04  1.688400e+04  1.688400e+04  1.688400e+04  


In [ ]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


Model 1: --- 31.622732218106588 minutes ---
The score is: 0.6616915422885572
The mean F1-Score (unweighted) is: 0.6539414347949808


Predicted Class,0,1,2,3
Realised Class,,,,
0,1521,243,298,41
1,194,328,154,5
2,256,175,721,13
3,19,20,10,223


              precision    recall  f1-score   support

           0       0.76      0.72      0.74      2103
           1       0.43      0.48      0.45       681
           2       0.61      0.62      0.61      1165
           3       0.79      0.82      0.81       272

    accuracy                           0.66      4221
   macro avg       0.65      0.66      0.65      4221
weighted avg       0.67      0.66      0.66      4221



In [ ]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

Model 2: --- 19.279711139202117 minutes ---
The score is: 0.8306088604596067
The mean F1-Score (unweighted) is: 0.8270387065938374


Predicted Class,0,1,2,3
Realised Class,,,,
0,1858,52,173,20
1,97,505,73,6
2,192,69,897,7
3,12,8,6,246


              precision    recall  f1-score   support

           0       0.86      0.88      0.87      2103
           1       0.80      0.74      0.77       681
           2       0.78      0.77      0.78      1165
           3       0.88      0.90      0.89       272

    accuracy                           0.83      4221
   macro avg       0.83      0.82      0.83      4221
weighted avg       0.83      0.83      0.83      4221



In [ ]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [ ]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

Model 3: --- 0.7346083998680115 minutes ---
The score is: 0.7986259180289031
The mean F1-Score (unweighted) is: 0.7857418348196589


Predicted Class,0,1,2,3
Realised Class,,,,
0,1858,52,160,33
1,161,407,108,5
2,240,56,866,3
3,11,8,13,240


              precision    recall  f1-score   support

           0       0.82      0.88      0.85      2103
           1       0.78      0.60      0.68       681
           2       0.76      0.74      0.75      1165
           3       0.85      0.88      0.87       272

    accuracy                           0.80      4221
   macro avg       0.80      0.78      0.79      4221
weighted avg       0.80      0.80      0.80      4221



In [ ]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [ ]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...

Model 4: --- 0.32759968042373655 minutes ---
The score is: 0.6481876332622601
The mean F1-Score (unweighted) is: 0.583693757437295


Predicted Class,0,1,2,3
Realised Class,,,,
0,1651,132,270,50
1,212,309,146,14
2,313,197,646,9
3,78,30,34,130


              precision    recall  f1-score   support

           0       0.73      0.79      0.76      2103
           1       0.46      0.45      0.46       681
           2       0.59      0.55      0.57      1165
           3       0.64      0.48      0.55       272

    accuracy                           0.65      4221
   macro avg       0.61      0.57      0.58      4221
weighted avg       0.64      0.65      0.64      4221

